In [20]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pprint import pprint

import balancepy as bp
from balancepy.data_class import sr_data as srd
from balancepy.model_sim.rfm25 import RFM25 as RFM25
from balancepy.model_sim.multi_model import MultiModel as MM

In [21]:
samplingrate_Hz = 100  # Hz
duration = 10        # seconds
t = np.arange(0, duration, 1/samplingrate_Hz)
freq = 0.2           # Hz
amplitude = 0.2
sine_wave = amplitude * np.sin(2 * np.pi * freq * t)

out = RFM25.velocity_deadzone(sine_wave, lamb=0.1, samplingrate_Hz=samplingrate_Hz)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=sine_wave, mode='lines', name='sine_wave'))
fig.add_trace(go.Scatter(x=t, y=out, mode='lines', name='velocity_deadzone'))
fig.update_layout(title='Sine Wave and Velocity Deadzone Output',
                  xaxis_title='Time (s)',
                  yaxis_title='Amplitude')
fig.show()

In [22]:

yi, yii, f = bp.spectrum(sine_wave, samplingrate_Hz)
yo, yoo, f = bp.spectrum(out, samplingrate_Hz)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=f[:20], y=np.abs(yi[:20]),
    mode='markers',
    marker=dict(symbol='circle', size=6),
    name='Sine Spectrum'
))
fig.add_trace(go.Scatter(
    x=f[:20], y=np.abs(yo[:20]),
    mode='markers',
    marker=dict(symbol='circle', size=6),
    name='Sine after deadzone Spectrum'
))
fig.update_layout(
    xaxis_title='Frequency (Hz)',
    yaxis_title='Magnitude',
    showlegend=True
)
fig.update_xaxes(type='log')
fig.show()

In [56]:
stim = bp.make_prts('Peterka2002',ampl=8,samplingrate_Hz = samplingrate_Hz)

L= 0.3

spec_deadzone = np.fft.fft(RFM25.velocity_deadzone(stim, L, samplingrate_Hz))
spec_no_deadzone = np.fft.fft(stim)

tmp = spec_deadzone[1:] / spec_no_deadzone[1:]

T = len(tmp) / samplingrate_Hz  # total time in seconds
start = 0
end = int(round(2.5 * T)) # frequencies up to 2.5 Hz
step = 2
selected_frequencies_index = np.arange(start, end, step)

f = np.fft.fftfreq(len(tmp), d=1/samplingrate_Hz)
f = f[selected_frequencies_index]

tmp = tmp[selected_frequencies_index]

fig = go.Figure()
fig.add_trace(go.Scatter(x=f, y=abs(tmp)))
fig.update_yaxes(range=[0, 1])